In [ ]:
import sys
import os

import numpy as np
import torch
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_pinball_loss
from gpytorch.mlls import VariationalELBO
from gpytorch_qr.likelihoods import MultitaskCenterGapQuantileGPLikelihood

sys.path.insert(0, os.path.abspath(".."))

try:
    import config_notebook
except ImportError:
    print("Output will not be deterministic SVG.")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def CV(x_train, y_train, x_test, y_test, quantiles, model, likelihood, n_epochs):
    mll = VariationalELBO(likelihood, model, num_data=y_train.shape[1])
    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(likelihood.parameters()),
        lr=0.001,
    )

    test_losses_per_fold = []
    for _ in range(n_epochs):
        model.train()
        likelihood.train()
        output = model(x_train)

        train_loss = -mll(output, y_train)
        train_loss.sum().backward()
        optimizer.step()
        optimizer.zero_grad()

        model.eval()
        likelihood.eval()
        with torch.no_grad():
            output = model.mean_quantiles_delta(x_test)  # (K, N, Q)
            epoch_fold_losses = []
            for y_test_fold, output_fold in zip(y_test, output):
                pinball_losses = []
                for j, q in enumerate(quantiles):
                    test_loss = mean_pinball_loss(
                        y_test_fold.cpu().numpy(),
                        output_fold[:, j].cpu().numpy(),
                        alpha=q.item(),
                    )
                    pinball_losses.append(test_loss)
                epoch_fold_losses.append(np.mean(pinball_losses))
            test_losses_per_fold.append(epoch_fold_losses)

    return np.array(test_losses_per_fold)

In [ ]:
QUANTILES = torch.tensor([0.05, 0.25, 0.5, 0.75, 0.95])
CENTER_QUANTILE_INDEX = 2
NUM_LOWER_QUANTILES = 2
NUM_LATENTS = 3
NUM_LOWER_LATENTS = 1
K = 5

N_EPOCHS = 1

In [ ]:
X = pd.read_csv("../_temp/X.csv").values
y = pd.read_csv("../_temp/y.csv")["H"].values

kf = KFold(n_splits=K, shuffle=True, random_state=42)
x_train_list, y_train_list, x_test_list, y_test_list = [], [], [], []
x_scales, x_mins = [], []
for train_idx, test_idx in kf.split(X):
    scaler = MinMaxScaler().fit(X[train_idx])

    x_train_list.append(torch.tensor(scaler.transform(X[train_idx])))
    y_train_list.append(torch.tensor(y[train_idx]))
    x_test_list.append(torch.tensor(scaler.transform(X[test_idx])))
    y_test_list.append(torch.tensor(y[test_idx]))

    x_scales.append(torch.tensor(scaler.scale_))
    x_mins.append(torch.tensor(scaler.min_))

x_train_cv = torch.stack(x_train_list).float().to(device)
y_train_cv = torch.stack(y_train_list).float().to(device)
x_test_cv = torch.stack(x_test_list).float().to(device)
y_test_cv = torch.stack(y_test_list).float().to(device)
x_scales = torch.stack(x_scales).float().to(device)
x_mins = torch.stack(x_mins).float().to(device)

In [ ]:
from scripts.model import MTGPQR_H

mtgpqr = MTGPQR_H(
    inducing_points=x_train_cv.clone().detach(),
    num_quantiles=len(QUANTILES),
    num_lower_quantiles=NUM_LOWER_QUANTILES,
    num_latents=NUM_LATENTS,
    num_lower_latents=NUM_LOWER_LATENTS,
    X_scale=x_scales,
    X_min=x_mins,
    batch_shape=torch.Size([K]),
).to(device)
likelihood = MultitaskCenterGapQuantileGPLikelihood(
    QUANTILES.unsqueeze(0),
    CENTER_QUANTILE_INDEX,
    torch.zeros((K, len(QUANTILES))),
    learn_scales=True,
).to(device)

mtgpqr_cv = CV(
    x_train_cv,
    y_train_cv,
    x_test_cv,
    y_test_cv,
    QUANTILES,
    mtgpqr,
    likelihood,
    n_epochs=N_EPOCHS,
)